# Phase 4 — export the cross-dimension comparison

`compare.physignal.z` has always run as part of Phase 4, but its result was only ever
printed to the console. That made the cross-dimension p-values README quotes the one set
of published numbers tracing to **no tracked file** — and with nothing to check them
against, a superseded version of that paragraph sat in README for five versions
undetected.

`src/r/phase4_kmult.R` now writes two files per run:

| file | what |
| --- | --- |
| `outputs/phase4/comparison_results.csv` | structured — one row per statistic per dimension pair |
| `outputs/phase4/comparison_summary.txt` | verbatim `print()` / `summary()` / `str()` of the object |

**The writer does not hard-code geomorph's field names.** There is no R runtime in this
project's local environment to verify them against, and this project has already shipped
one bug from a guessed geomorph field name (v4.1.1, `P.value`). So it walks whatever the
returned object actually contains and records the names it finds. If it finds nothing
numeric it says so loudly and still writes the summary text — read its `str()` section
and extend `write_comparison_output()` to match.

**Use a CPU runtime.** No GPU needed. The slow step is installing `geomorph`
(5–15 minutes, compiles from source).

Everything else here is a re-run of the existing deterministic Phase 4 script
(`physignal.z` runs with `seed = NULL`, documented as deterministic; the Mantel step
uses `set.seed(1)`), so the already-published result files should come back
**unchanged**. The second-to-last cell checks exactly that — if they move, something
else has drifted and that is worth knowing before you commit anything.

## Setup

> **If you hit `Transport endpoint is not connected`** — that is the Google Drive
> FUSE mount collapsing, not a git problem, and it can corrupt the repo on Drive if
> it happens mid-write. It cannot be fixed from inside the same session:
>
> 1. **Runtime → Restart session**
> 2. Re-run the mount cell
> 3. Run the **Recovery** cell below, which verifies the repo and re-clones it if
>    it is damaged
>
> Nothing important lives only on Drive — every result file is in the repo. The one
> exception is `data/extracted_fish/` (~480 MB, gitignored), which the recovery cell
> preserves. **This notebook does not need it at all**, so the re-run can proceed even
> if those images are lost.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/guptrishi01/Surgeonfish_Neural_Network_Phylogenetics.git"
PROJECT_DIR = Path("/content/drive/MyDrive/Surgeonfish_Neural_Network_Phylogenetics")


def git(*args, check=True):
    """Runs git and always shows its real output - a bare check=True hides it."""
    r = subprocess.run(["git", "-C", str(PROJECT_DIR), *args],
                       capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.stderr.strip():
        print(r.stderr.strip())
    if check and r.returncode:
        raise RuntimeError(f"git {' '.join(args)} failed ({r.returncode}) - see above")
    return r


if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    # This clone predates outputs/ and reports/ being tracked in git, so it
    # holds untracked local copies of files that now exist in the repo, and a
    # plain pull aborts on them. They are all either identical to the tracked
    # versions or stale (the repo was re-run after they were made), so the fix
    # is to move them aside - to a timestamped backup folder, not /dev/null -
    # and pull again.
    import re, shutil, datetime

    result = git("pull", check=False)
    if result.returncode:
        blocked = re.findall(r"^\t(.+)$", result.stderr, flags=re.M)
        if not blocked:
            raise RuntimeError("pull failed for a reason other than blocking files - see above")
        stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        backup = PROJECT_DIR.parent / f"surgeonfish-prepull-backup-{stamp}"
        print(f"\n{len(blocked)} untracked file(s) blocking the pull.")
        print(f"Moving them to {backup} (nothing deleted), then retrying...\n")
        for rel in blocked:
            src = PROJECT_DIR / rel
            if not src.exists():
                continue
            dest = backup / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(dest))
        git("pull")

git("log", "--oneline", "-1")

### Recovery — run this if the mount dropped or the pull left the repo broken

Checks the repo is readable and consistent (`git fsck`). If it isn't, moves
`data/extracted_fish/` aside, re-clones from scratch, and puts the images back.

In [ ]:
import shutil, subprocess
from pathlib import Path

if not Path("/content/drive/MyDrive").is_dir():
    raise SystemExit(
        "Drive is not mounted. Runtime -> Restart session, then re-run the mount cell."
    )

def repo_is_healthy(path):
    if not (path / ".git").is_dir():
        return False
    probe = subprocess.run(["git", "-C", str(path), "rev-parse", "HEAD"],
                           capture_output=True, text=True)
    if probe.returncode:
        return False
    fsck = subprocess.run(["git", "-C", str(path), "fsck", "--connectivity-only"],
                          capture_output=True, text=True)
    return fsck.returncode == 0

if repo_is_healthy(PROJECT_DIR):
    print("Repo looks healthy - no recovery needed.")
else:
    print("Repo is missing or damaged. Re-cloning...\n")
    images = PROJECT_DIR / "data/extracted_fish"
    parked = PROJECT_DIR.parent / "extracted_fish_rescued"
    if images.is_dir() and not parked.exists():
        print("Preserving data/extracted_fish (this takes a minute)...")
        shutil.move(str(images), str(parked))
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    if parked.is_dir():
        (PROJECT_DIR / "data").mkdir(parents=True, exist_ok=True)
        shutil.move(str(parked), str(PROJECT_DIR / "data/extracted_fish"))
        print("Restored data/extracted_fish")
    print("\nRe-clone complete.")

subprocess.run(["git", "-C", str(PROJECT_DIR), "log", "--oneline", "-1"])

In [ ]:
%cd {PROJECT_DIR}

In [ ]:
%pip install -q "biopython>=1.81"
import sys
sys.path.insert(0, str(PROJECT_DIR / "src"))
print("ready")

## Rebuild the Kmult-ready inputs

Same two calls Part A of `Followups.ipynb` makes — the primary 49-species set and the
47-species sensitivity set — so the R script has both to run against. Phases 2 and 3
are already re-run and tracked in the repo you just pulled, so this only rebuilds the
matrices R needs and does **not** need the ~480 MB of extracted images.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-8s %(message)s", force=True)

from phylo_comparison.config import ExportConfig, FeaturePrepConfig
from phylo_comparison.pipeline import run as run_prep

species_order = run_prep(
    FeaturePrepConfig(species_features_csv_path=PROJECT_DIR / "reports/species_features.csv"),
    ExportConfig(
        output_dir=PROJECT_DIR / "outputs/phase4",
        tree_path=PROJECT_DIR / "data/phylogeny/actinopt_12k_treePL.tre",
        species_coverage_csv_path=PROJECT_DIR / "data/phylogeny/species_coverage.csv",
    ),
)
print(f"prepared {len(species_order)} species")

In [ ]:
# Same for the 47-species sensitivity set, so the R script runs both.
# Phases 2 and 3 were already re-run locally and their outputs are in the
# repo, so this only rebuilds the Kmult-ready matrices - no need for the
# mask PNGs that Phase 3 aggregation would otherwise require.
run_prep(
    FeaturePrepConfig(
        species_features_csv_path=PROJECT_DIR / "reports/species_features_min5.csv"
    ),
    ExportConfig(
        output_dir=PROJECT_DIR / "outputs/phase4_min5",
        tree_path=PROJECT_DIR / "data/phylogeny/actinopt_12k_treePL.tre",
        species_coverage_csv_path=PROJECT_DIR / "data/phylogeny/species_coverage.csv",
    ),
)
print("sensitivity set prepared")

## Install R + `geomorph` and run the script

This sources the actual version-controlled script rather than a copy pasted into the
notebook, so there is exactly one place this logic lives.

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
# geomorph compiles from source on first install - 5-15 minutes. Be patient.
if (!requireNamespace("geomorph", quietly = TRUE)) {
  install.packages("geomorph", dependencies = TRUE, Ncpus = 2)
}
library(geomorph)
packageVersion("geomorph")

In [ ]:
project_dir_str = str(PROJECT_DIR)

In [ ]:
%%R -i project_dir_str
setwd(project_dir_str)
source("src/r/phase4_kmult.R")

## What came out

In [ ]:
import pandas as pd

phase4_dir = PROJECT_DIR / "outputs/phase4"
csv_path = phase4_dir / "comparison_results.csv"
txt_path = phase4_dir / "comparison_summary.txt"

if csv_path.exists():
    print(f"{csv_path.name} — the file README's cross-dimension numbers will "
          f"be checked against:")
    display(pd.read_csv(csv_path))
else:
    print("comparison_results.csv was NOT written: the structured walk found no")
    print("numeric fields on the returned object. This is the documented fallback,")
    print("not a crash. Read the str() section printed below, then extend")
    print("write_comparison_output() in src/r/phase4_kmult.R to match the real shape.")

print(f"\n{'=' * 70}\n{txt_path.name}\n{'=' * 70}")
print(txt_path.read_text(encoding="utf-8"))

## Did re-running change anything that was already published?

It should not. New files are the point of this run; **modified** ones are not, and
mean a previously published number moved.

In [ ]:
import subprocess

status = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "status", "--porcelain", "outputs/", "reports/"],
    capture_output=True, text=True,
).stdout.splitlines()

new = [line for line in status if line.startswith("??")]
modified = [line for line in status if line.strip() and not line.startswith("??")]

print("New files (expected):")
for line in new or ["  (none)"]:
    print("  ", line)

print("\nModified existing results (expected: none):")
for line in modified or ["  (none)"]:
    print("  ", line)

if modified:
    print("\n*** A previously published result changed. Inspect before committing:")
    print("      git diff outputs/ reports/")
else:
    print("\nClean - the re-run reproduced every existing result.")

## Get the files into the repo

Easiest path is to download them and commit from your local clone (Colab has no push
credentials by default):

In [ ]:
from google.colab import files

for path in (csv_path, txt_path):
    if path.exists():
        files.download(str(path))

Then, in your local clone:

```bash
git add outputs/phase4/comparison_results.csv outputs/phase4/comparison_summary.txt
git commit -m "Track compare.physignal.z output (closes follow-up 6)"
git push
```

Two things follow automatically once those are committed:

- `pytest -k cross_dimension` stops skipping and starts asserting that every
  cross-dimension p-value README quotes actually appears in the file.
- If the fresh numbers differ from the `p`=0.040 / 0.088 currently in README, that
  test goes red — which is the whole point. Update the README paragraph to whatever
  the file says, rather than the other way round.